In [ ]:
import polars as pl
import pandas as pd
from definitions import ROOT_DIR
from datetime import date

start_date = date(2025, 7, 1)
end_date = date(2025, 7, 1)

def get_data_path(d: date) -> str:
    return str(ROOT_DIR / f"data/orderbook/binance_futures/BTCUSDT/{d.strftime('%Y-%m-%d')}.parquet")

DATA_PATHS = [get_data_path(d) for d in pd.date_range(start=start_date, end=end_date)]

df = pl.read_parquet(DATA_PATHS)

In [ ]:
df.head()

In [ ]:
# df = df.sort("received_time")
# df

In [ ]:
# Convert event_time from milliseconds to datetime and extract hour
df = df.with_columns(
    (pl.col("event_time").cast(pl.Datetime("ms")).dt.day()).alias("day"),
    (pl.col("event_time").cast(pl.Datetime("ms")).dt.hour()).alias("hour"),
    (pl.col("event_time").cast(pl.Datetime("ms")).dt.minute()).alias("minute"),
    (pl.col("event_time").cast(pl.Datetime("ms")).dt.second()).alias("second")
)


# Group by day and hour, and count the number of events
events_per_day_hour = df.group_by(['day', 'hour']).agg(pl.len().alias('count')).sort(['day', 'hour'])

events_per_day_hour.collect()
# # Plot the histogram for each day
# events_per_day_hour.hvplot.bar(x='hour', y='count', by='day', title='Number of Events per Hour for Each Day', xlabel='Hour', ylabel='Number of Events')
